# Topic 19b - LLM gateways

Companion notebook for `gateway.py`: one OpenAI-shaped endpoint fanning out to
two providers, per-key budgets/limits, dynamic routes and guardrails.

## 0. Import + a deterministic provider caller

In [1]:
from gateway import (
    BudgetLimitNode,
    ConditionalNode,
    CompletionResult,
    DynamicRoute,
    Gateway,
    GatewayError,
    ModelDeployment,
    ModelNode,
    PIIMasker,
    PercentageNode,
    RateLimitNode,
    VirtualKey,
    recommend_gateway,
)

MODELS = [
    ModelDeployment("gpt-small", "openai", input_cost_per_token=1e-6, output_cost_per_token=2e-6),
    ModelDeployment("claude-big", "anthropic", input_cost_per_token=5e-6, output_cost_per_token=15e-6),
]


def caller(dep, messages, prompt_tokens):
    return CompletionResult(f"[{dep.provider}] {messages[-1]['content']}", prompt_tokens or 5, 10)


print("imported gateway.py")

imported gateway.py


## 1. One endpoint, two providers

The client sends one OpenAI-shaped request to a model name; the gateway fans it
out to the owning provider and restamps the answer.

In [2]:
gw = Gateway(MODELS, caller)
gw.add_key(VirtualKey("sk-team", owner="alice", team="eng", allowed_models={"gpt-small", "claude-big"}))

resp = gw.completion(api_key="sk-team", model="claude-big",
                     messages=[{"role": "user", "content": "hello"}], prompt_tokens=8)
print("model:", resp["model"])
print("content:", resp["choices"][0]["message"]["content"])
print("usage:", resp["usage"])
print("log:", gw.logs[-1])

model: claude-big
content: [anthropic] hello
usage: {'prompt_tokens': 8, 'completion_tokens': 10, 'total_tokens': 18}
log: LogRecord(key='sk-team', team='eng', model='claude-big', provider='anthropic', input_tokens=8, output_tokens=10, cost=0.00019, route=None, tags=[])


## 2. Virtual keys: access, budget, rate limit

In [3]:
locked = Gateway(MODELS, caller)
locked.add_key(VirtualKey("sk-small", owner="bob", team="qa", allowed_models={"gpt-small"}))
try:
    locked.completion(api_key="sk-small", model="claude-big",
                      messages=[{"role": "user", "content": "hi"}])
except GatewayError as exc:
    print("model access blocked ->", exc.status, exc.code)

broke = Gateway(MODELS, caller)
broke.add_key(VirtualKey("sk-broke", owner="bob", team="qa", allowed_models={"gpt-small"}, max_budget=0.0))
try:
    broke.completion(api_key="sk-broke", model="gpt-small",
                     messages=[{"role": "user", "content": "hi"}], prompt_tokens=100)
except GatewayError as exc:
    print("budget blocked ->", exc.status, exc.code)

busy = Gateway(MODELS, caller)
busy.add_key(VirtualKey("sk-busy", owner="bob", team="qa", allowed_models={"gpt-small"}, rpm_limit=1))
busy.completion(api_key="sk-busy", model="gpt-small", messages=[{"role": "user", "content": "a"}])
try:
    busy.completion(api_key="sk-busy", model="gpt-small", messages=[{"role": "user", "content": "b"}])
except GatewayError as exc:
    print("rate limit blocked ->", exc.status, exc.code)

model access blocked -> 403 model_not_allowed
budget blocked -> 402 budget_exceeded
rate limit blocked -> 429 rate_limit_exceeded


## 3. Spend tracking per key, team and tag

In [4]:
tracked = Gateway(MODELS, caller)
tracked.add_key(VirtualKey("sk-x", owner="alice", team="eng", allowed_models={"claude-big"}))
tracked.completion(api_key="sk-x", model="claude-big",
                   messages=[{"role": "user", "content": "hi"}], prompt_tokens=10, tags=["trial"])
key = tracked.keys["sk-x"]
print("key.spend      :", round(key.spend, 6))
print("team spend     :", round(tracked.team_spend["eng"], 6))
print("tag spend      :", round(key.tag_spend["trial"], 6))

key.spend      : 0.0002
team spend     : 0.0002
tag spend      : 0.0002


## 4. Dynamic routes: conditional, percentage, limits

A route is a versioned flow that picks a model from request metadata or key
state. Change it without touching application code.

In [5]:
cond = Gateway(MODELS, caller)
cond.routes.publish(DynamicRoute("support", [
    ConditionalNode(lambda ctx: ctx.metadata.get("user_plan") == "paid", "claude-big", "gpt-small"),
]), "v1")
cond.routes.deploy("support", "v1")
cond.add_key(VirtualKey("sk-r", owner="a", team="eng", allowed_models={"gpt-small", "claude-big"}))
print("paid ->", cond.completion(api_key="sk-r", model="support",
      messages=[{"role": "user", "content": "x"}], metadata={"user_plan": "paid"})["model"])
print("free ->", cond.completion(api_key="sk-r", model="support",
      messages=[{"role": "user", "content": "x"}], metadata={"user_plan": "free"})["model"])

paid -> claude-big
free -> gpt-small


In [6]:
# Percentage (A/B): deterministic with a seeded rng
ab = Gateway(MODELS, caller, rng=lambda: 0.95)
ab.routes.publish(DynamicRoute("ab", [PercentageNode([("gpt-small", 0.9), ("claude-big", 0.1)])]), "v1")
ab.routes.deploy("ab", "v1")
ab.add_key(VirtualKey("sk-ab", owner="a", team="eng", allowed_models={"gpt-small", "claude-big"}))
print("rng=0.95 ->", ab.completion(api_key="sk-ab", model="ab", messages=[{"role": "user", "content": "x"}])["model"])

rng=0.95 -> claude-big


In [7]:
# Rate-limit node switches to a cheap model once the quota is hit
rl = Gateway(MODELS, caller)
rl.routes.publish(DynamicRoute("support", [RateLimitNode("support", 60.0, 1, "claude-big", "gpt-small")]), "v1")
rl.routes.deploy("support", "v1")
rl.add_key(VirtualKey("sk-rl", owner="a", team="eng", allowed_models={"gpt-small", "claude-big"}))
print("1st:", rl.completion(api_key="sk-rl", model="support", messages=[{"role": "user", "content": "a"}])["model"])
print("2nd:", rl.completion(api_key="sk-rl", model="support", messages=[{"role": "user", "content": "b"}])["model"])

1st: claude-big
2nd: gpt-small


## 5. Versions + rollback (no code change)

In [8]:
reg = Gateway(MODELS, caller)
reg.routes.publish(DynamicRoute("r", [ModelNode("gpt-small")]), "v1")
reg.routes.publish(DynamicRoute("r", [ModelNode("claude-big")]), "v2")
reg.add_key(VirtualKey("sk-v", owner="a", team="eng", allowed_models={"gpt-small", "claude-big"}))
for version in ("v1", "v2", "v1"):
    if version == "v1":
        reg.routes.rollback("r", "v1")
    else:
        reg.routes.deploy("r", version)
    print(version, "->", reg.completion(api_key="sk-v", model="r", messages=[{"role": "user", "content": "x"}])["model"])

v1 -> gpt-small
v2 -> claude-big
v1 -> gpt-small


## 6. Guardrails: PII masking on input and output

In [9]:
masked = Gateway(MODELS, caller, guardrails=[PIIMasker()])
masked.add_key(VirtualKey("sk-p", owner="a", team="eng", allowed_models={"gpt-small"}))
out = masked.completion(api_key="sk-p", model="gpt-small",
                        messages=[{"role": "user", "content": "reach me at a.b@example.com"}])
print("masked content:", out["choices"][0]["message"]["content"])

masked content: [openai] reach me at [email]


## 7. Gateway vs SDK decision

In [10]:
for args in [(1, {}), (3, {}), (1, {"spend_visibility": True}), (1, {"guardrails": True})]:
    decision, why = recommend_gateway(args[0], **args[1])
    print(f"apps={args[0]} {args[1]} -> {decision}: {why}")

apps=1 {} -> sdk: one app, one key, no shared controls: call the provider SDK directly
apps=3 {} -> gateway: use a gateway: 3 apps
apps=1 {'spend_visibility': True} -> gateway: use a gateway: spend visibility
apps=1 {'guardrails': True} -> gateway: use a gateway: guardrails


## 8. Notes

- Past one app/person, or when keys/budgets/spend-logs/guardrails matter, use a
  gateway; below that, the provider SDK is simpler.
- Dynamic routes are versioned: publish a draft, deploy, roll back instantly.
- Always re-check budget/access against the model that actually bills.

Run the suite: `python -m pytest -q` in this folder.